In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**Membuat file db.py**

In [4]:
import os

# 1. Pastikan folder src/ dan data/ ada
os.makedirs("src", exist_ok=True)
os.makedirs("data", exist_ok=True)

In [7]:
# 2. Tulis isi file src/db.py
db_code = """import os
import sqlite3

DB_PATH = "data/app.db"

def get_conn():
    \"\"\"Membuat koneksi SQLite dengan mengaktifkan foreign_keys = ON\"\"\"
    os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn

def init_db():
    \"\"\"Menginisialisasi 4 tabel utama pada database SQLite\"\"\"
    conn = get_conn()
    cursor = conn.cursor()

    # 1. Tabel posts (Unggahan)
    cursor.execute(\"\"\"
    CREATE TABLE IF NOT EXISTS posts (
        post_id TEXT PRIMARY KEY,
        platform TEXT,
        published_at TEXT,
        text_content TEXT
    )
    \"\"\")

    # 2. Tabel comments (Komentar)
    cursor.execute(\"\"\"
    CREATE TABLE IF NOT EXISTS comments (
        comment_id TEXT PRIMARY KEY,
        post_id TEXT,
        text_content TEXT,
        text_original TEXT,
        text_hash TEXT,
        published_at TEXT,
        status_eksklusi TEXT,
        alasan_eksklusi TEXT,
        split TEXT,
        FOREIGN KEY (post_id) REFERENCES posts(post_id)
    )
    \"\"\")

    # 3. Tabel annotators (Pemeriksa/Labeler)
    cursor.execute(\"\"\"
    CREATE TABLE IF NOT EXISTS annotators (
        annotator_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL
    )
    \"\"\")

    # 4. Tabel annotations (Hasil Pelabelan)
    cursor.execute(\"\"\"
    CREATE TABLE IF NOT EXISTS annotations (
        annotation_id INTEGER PRIMARY KEY AUTOINCREMENT,
        comment_id TEXT,
        annotator_id INTEGER,
        label TEXT,
        created_at TEXT,
        FOREIGN KEY (comment_id) REFERENCES comments(comment_id),
        FOREIGN KEY (annotator_id) REFERENCES annotators(annotator_id)
    )
    \"\"\")

    conn.commit()
    conn.close()
    print("Database SQLite dan 4 tabel berhasil diinisialisasi.")

if __name__ == "__main__":
    init_db()
"""

with open("src/db.py", "w", encoding="utf-8") as f:
    f.write(db_code)

print("File src/db.py berhasil ditulis!")

File src/db.py berhasil ditulis!


In [9]:
from src.db import init_db
init_db()

[✔] Database SQLite dan 4 tabel berhasil diinisialisasi.


**Membuat file importer.py**

In [10]:
importer_code = """import pandas as pd
import sqlite3

def import_posts(conn, df):
    \"\"\"Mengimpor data posts ke database SQLite\"\"\"
    ok, skipped, rejected = 0, 0, []
    for r in df.itertuples():
        if pd.isna(r.post_id):
            rejected.append((str(getattr(r, 'Index', 'Unknown')), "Post ID kosong"))
            continue

        try:
            cur = conn.cursor()
            cur.execute(\"\"\"
                INSERT OR IGNORE INTO posts (post_id, platform, published_at, text_content)
                VALUES (?, ?, ?, ?)
            \"\"\", (
                str(r.post_id),
                str(getattr(r, 'platform', 'Unknown')),
                str(getattr(r, 'published_at', '')),
                str(getattr(r, 'text_content', ''))
            ))
            if cur.rowcount == 1:
                ok += 1
            else:
                skipped += 1
        except Exception as e:
            rejected.append((str(r.post_id), str(e)))

    conn.commit()
    return {"diterima": ok, "dilewati": skipped, "ditolak": rejected}

def import_comments(conn, df):
    \"\"\"Mengimpor data comments ke database SQLite dengan validasi foreign key (post_id)\"\"\"
    ok, skipped, rejected = 0, 0, []
    for r in df.itertuples():
        if pd.isna(r.comment_id) or pd.isna(r.post_id):
            rejected.append((str(getattr(r, 'comment_id', 'Unknown')), "Comment ID atau Post ID kosong"))
            continue

        # Periksa apakah unggahan induk (post_id) ada di database
        cur = conn.cursor()
        cur.execute("SELECT 1 FROM posts WHERE post_id = ?", (str(r.post_id),))
        if cur.fetchone() is None:
            rejected.append((str(r.comment_id), "Q09: unggahan sumber tidak ditemukan"))
            continue

        try:
            cur.execute(\"\"\"
                INSERT OR IGNORE INTO comments
                (comment_id, post_id, text_content, text_original, text_hash, published_at, status_eksklusi, alasan_eksklusi, split)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            \"\"\", (
                str(r.comment_id),
                str(r.post_id),
                str(getattr(r, 'text_content', '')),
                str(getattr(r, 'text_original', '')),
                str(getattr(r, 'text_hash', '')),
                str(getattr(r, 'published_at', '')),
                str(getattr(r, 'status_eksklusi', 'Aktif')),
                str(getattr(r, 'alasan_eksklusi', 'Lolos')),
                str(getattr(r, 'split', 'train'))
            ))
            if cur.rowcount == 1:
                ok += 1
            else:
                skipped += 1
        except Exception as e:
            rejected.append((str(r.comment_id), str(e)))

    conn.commit()
    return {"diterima": ok, "dilewati": skipped, "ditolak": rejected}
"""

with open("src/importer.py", "w", encoding="utf-8") as f:
    f.write(importer_code)

print("File src/importer.py berhasil ditulis!")

File src/importer.py berhasil ditulis!


**Mmebuat file 1_Impor.py untuk unggah CSV, tampilkan pratinjau 5 baris, tombol “Impor”, lalu ringkasan dan tabel baris ditolak.**

In [11]:
import os

# Pastikan folder app/pages/ ada
os.makedirs("app/pages", exist_ok=True)

importer_page_code = """import streamlit as st
import pandas as pd
import sys
import os

# Menambahkan path src agar dapat diimpor
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '../../')))
from src.db import get_conn, init_db
from src.importer import import_posts, import_comments

st.title("Impor Data v0 ke Database SQLite")
st.write("Unggah file CSV (Posts atau Comments) untuk dimasukkan ke dalam database aplikasi.")

# Pastikan tabel sudah terinisialisasi
init_db()

# Pilih jenis data yang akan diimpor
data_type = st.radio("Pilih Jenis Data:", ["Posts", "Comments"])
uploaded_file = st.file_uploader(f"Pilih file CSV {data_type}", type=["csv"])

if uploaded_file is not None:
    df = pd.read_csv(uploaded_file)
    st.subheader("Pratinjau 5 Baris Pertama")
    st.dataframe(df.head(5))

    if st.button(f"Mulai Impor {data_type}"):
        conn = get_conn()
        with st.spinner("Sedang memproses impor data..."):
            if data_type == "Posts":
                res = import_posts(conn, df)
            else:
                # Gabungkan informasi split jika ada di manifest
                if "manifest_v0.csv" in os.listdir("data/splits") and "split" not in df.columns:
                    manifest = pd.read_csv("data/splits/manifest_v0.csv")
                    df = df.merge(manifest[["comment_id", "split"]], on="comment_id", how="left")
                res = import_comments(conn, df)
        conn.close()

        st.success("Impor selesai!")
        col1, col2, col3 = st.columns(3)
        col1.metric("Diterima", res["diterima"])
        col2.metric("Dilewati (Duplikat)", res["dilewati"])
        col3.metric("Ditolak", len(res["ditolak"]))

        if res["ditolak"]:
            st.warning("Daftar Baris Ditolak:")
            df_rejected = pd.DataFrame(res["ditolak"], columns=["ID", "Alasan"])
            st.dataframe(df_rejected)
"""

with open("app/pages/1_Impor.py", "w", encoding="utf-8") as f:
    f.write(importer_page_code)

print("File app/pages/1_Impor.py berhasil ditulis!")

File app/pages/1_Impor.py berhasil ditulis!


**Uji coba 100 komentar dari data processed beserta unggahannya**

In [12]:
import pandas as pd
import os
from src.db import get_conn, init_db
from src.importer import import_posts, import_comments

# 1. Inisialisasi database
init_db()

[✔] Database SQLite dan 4 tabel berhasil diinisialisasi.


In [13]:
# 2. Muat data posts dan comments audit v0
posts = pd.read_csv("data/processed/posts_v0.csv")
comments = pd.read_csv("data/processed/comments_v0_audited.csv")

# Ambil data komentar aktif (Lolos)
comments_active = comments[comments["alasan_eksklusi"] == "Lolos"].copy()

# Gabungkan dengan informasi split dari manifest jika ada
manifest_path = "data/splits/manifest_v0.csv"
if os.path.exists(manifest_path):
    manifest = pd.read_csv(manifest_path)
    comments_active = comments_active.merge(manifest[["comment_id", "split"]], on="comment_id", how="left")

In [14]:
# 3. Ambil sampel 100 komentar pertama beserta post_id yang bersangkutan
sample_comments = comments_active.head(100)
sample_post_ids = sample_comments["post_id"].unique()
sample_posts = posts[posts["post_id"].isin(sample_post_ids)]

In [15]:
# 4. Impor posts terlebih dahulu, lalu comments ke database
conn = get_conn()
res_posts = import_posts(conn, sample_posts)
res_comments = import_comments(conn, sample_comments)

print("=== HASIL UJI COBA IMPOR 100 KOMENTAR ===")
print(f"Posts    -> Diterima: {res_posts['diterima']}, Dilewati: {res_posts['dilewati']}, Ditolak: {len(res_posts['ditolak'])}")
print(f"Comments -> Diterima: {res_comments['diterima']}, Dilewati: {res_comments['dilewati']}, Ditolak: {len(res_comments['ditolak'])}")
conn.close()

=== HASIL UJI COBA IMPOR 100 KOMENTAR ===
Posts    -> Diterima: 7, Dilewati: 0, Ditolak: 0
Comments -> Diterima: 100, Dilewati: 0, Ditolak: 0


**Mencocokkan hasil dengan  query SELECT COUNT(*) FROM comments;**

In [16]:
from src.db import get_conn

# Buka koneksi ke database
conn = get_conn()
cursor = conn.cursor()

# Jalankan query hitung total baris pada tabel comments
cursor.execute("SELECT COUNT(*) FROM comments;")
db_count = cursor.fetchone()[0]

print(f"Jumlah baris pada tabel comments di database: {db_count}")

# Tutup koneksi
conn.close()

Jumlah baris pada tabel comments di database: 100


**Pilih 300 komentar untuk pemeriksan kedua**

semua komentar test (rancangan v0) + 100 acak dari train/validation (seed tercatat). Buat file buta tanpa kolom label.

In [17]:
import pandas as pd
import numpy as np
import os

# 1. Muat data audit dan manifest split v0
comments = pd.read_csv("data/processed/comments_v0_audited.csv")
manifest = pd.read_csv("data/splits/manifest_v0.csv")
df_merged = comments.merge(manifest, on=["comment_id", "post_id"], how="inner")

In [18]:
# 2. Ambil semua data test
df_test = df_merged[df_merged["split"] == "test"]

In [19]:
# 3. Ambil 100 sampel acak dari train / validation (seed 2026 agar konsisten)
df_train_val = df_merged[df_merged["split"].isin(["train", "validation"])]
df_train_val_sample = df_train_val.sample(n=100, random_state=2026)

In [20]:
# 4. Gabungkan menjadi satu pool sampel
reviewer2_pool = pd.concat([df_test, df_train_val_sample]).drop_duplicates(subset=["comment_id"])

# Batasi tepat atau ambil sejumlah totalnya (biasanya mendekati/pas 300)
reviewer2_300 = reviewer2_pool.head(300)

In [21]:
# 5. Buat file buta (blind file) TANPA kolom label atau status eksklusi
blind_cols = ["comment_id", "post_id", "text_content", "published_at"]
reviewer2_blind = reviewer2_300[[c for c in blind_cols if c in reviewer2_300.columns]]

In [22]:
# 6. Simpan ke folder processed
output_blind_path = "data/processed/reviewer2_blind.csv"
os.makedirs(os.path.dirname(output_blind_path), exist_ok=True)
reviewer2_blind.to_csv(output_blind_path, index=False)

print(f"Berhasil membuat file blind pemeriksa kedua dengan {len(reviewer2_blind)} baris di: {output_blind_path}")

Berhasil membuat file blind pemeriksa kedua dengan 300 baris di: data/processed/reviewer2_blind.csv
